In [2]:
!pip install -q langgraph langchain-groq langchain-community langchain-text-splitters chromadb sentence-transformers pypdf python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9

In [3]:


# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# Typing
from typing import Annotated
from typing_extensions import TypedDict

# LangChain
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

# File handling
from google.colab import files, userdata
from docx import Document
from pypdf import PdfReader
import io

print("✅ All imports done!")

/tmp/ipykernel_2281/2019295837.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


✅ All imports done!


setting up LLM using secret Key

In [4]:
llm = ChatGroq(
    api_key=userdata.get('GroqKey'),
    model="llama-3.3-70b-versatile"
)

print("✅ LLM ready!")

✅ LLM ready!


In [5]:
class InterviewState(TypedDict):
    # Documents
    cv_text: str
    jd_text: str

    # Node 1 output
    extracted_skills: str

    # Node 2 output
    retrieved_context: str

    # Node 3 output
    current_question: str
    difficulty: str
    asked_questions: list

    # User input
    user_answer: str
    # Node 4 output
    score: int
    feedback: str
    hint: str

    # Progress
    question_count: int
    total_score: int

    # Node 5 output
    final_report: str

    # Memory
    messages: Annotated[list, add_messages]

In [6]:
print("📄 Step 1: Upload your CV file (.docx, .pdf, or .txt)")
uploaded_cv = files.upload()

cv_filename = list(uploaded_cv.keys())[0]
cv_content = uploaded_cv[cv_filename]

# Read CV based on file type
if cv_filename.endswith(".txt"):
    cv_text = cv_content.decode("utf-8")
elif cv_filename.endswith(".pdf"):
    from pypdf import PdfReader
    reader = PdfReader(io.BytesIO(cv_content))
    cv_text = ""
    for page in reader.pages:
        cv_text += page.extract_text()
elif cv_filename.endswith(".docx"):
    doc = Document(io.BytesIO(cv_content))
    cv_text = ""
    for paragraph in doc.paragraphs:
        cv_text += paragraph.text + "\n"

print(f"✅ CV uploaded: {cv_filename}")
print(f"\n📝 CV Preview:\n{cv_text[:300]}...")
print("📋 Step 2: Upload Job Description file (.docx, .pdf, or .txt)")
uploaded_jd = files.upload()

jd_filename = list(uploaded_jd.keys())[0]
jd_content = uploaded_jd[jd_filename]

# Read job description based on file type
if jd_filename.endswith(".txt"):
    jd_text = jd_content.decode("utf-8")
elif jd_filename.endswith(".pdf"):
    from pypdf import PdfReader
    reader = PdfReader(io.BytesIO(jd_content))
    jd_text = ""
    for page in reader.pages:
        jd_text += page.extract_text()
elif jd_filename.endswith(".docx"):
    doc = Document(io.BytesIO(jd_content))
    jd_text = ""
    for paragraph in doc.paragraphs:
        jd_text += paragraph.text + "\n"

print(f"✅ Job Description uploaded: {jd_filename}")
print(f"\n📝 Job Description Preview:\n{jd_text[:300]}...")

📄 Step 1: Upload your CV file (.docx, .pdf, or .txt)


Saving Sonaina_Ilyas_Resume_Simera.docx to Sonaina_Ilyas_Resume_Simera.docx
✅ CV uploaded: Sonaina_Ilyas_Resume_Simera.docx

📝 CV Preview:
Sonaina Ilyas
Islamabad, Pakistan (Open to Remote)  |  sonainailyas005@gmail.com  |  +92 324 9752902
linkedin.com/in/sonaina-ilyas-3a0b38331
PROFESSIONAL SUMMARY

Organized, detail-focused administrative professional with an MS in Computer Science and three years of running the day-to-day administra...
📋 Step 2: Upload Job Description file (.docx, .pdf, or .txt)


Saving jobDes.txt to jobDes.txt
✅ Job Description uploaded: jobDes.txt

📝 Job Description Preview:
Job Title: Data Analyst
Company: ABC Company

Requirements:
- 2+ years experience in data analysis
- Python and SQL skills
- Experience with Tableau or Power BI
- Strong communication skills
- AI and Machine Learning knowledge preferred

Responsibilities:
- Analyze large datasets
- Creat...


In [7]:
from re import search
from langchain_core.vectorstores import VectorStore
from importlib import metadata
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
cv_chunks= text_splitter.create_documents(
    texts=[cv_text],
    metadatas=[{"source": "cv"}]
)
jd_chunks=text_splitter.create_documents(
    texts=[jd_text],
    metadatas=[{"source": "jd"}]
)
all_chunks=cv_chunks+jd_chunks

embeddings=SentenceTransformerEmbeddings(
     model_name="all-MiniLM-L6-v2"
)
vectorstore= Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
)

retriever= vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"✅ CV split into {len(cv_chunks)} chunks!")
print(f"✅ JD split into {len(jd_chunks)} chunks!")
print("✅ RAG ready!")

/tmp/ipykernel_2281/1674765912.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=SentenceTransformerEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ CV split into 14 chunks!
✅ JD split into 1 chunks!
✅ RAG ready!


In [8]:
def analyze_resume(state: InterviewState):
    print("📄 Node 1: Analyzing resume...")
    try:
        prompt = f"""
        You are a senior recruiter.
        Analyze the CV based on {state['cv_text']}
        and the job description {state['jd_text']}.
        Extract skills based on the job description and find gaps
        between the job description and the CV.
        """
        response = llm.invoke(prompt)
        return {
            "extracted_skills": response.content,
            "difficulty": "medium"
        }
    except Exception as e:
        print(f"❌ Error in Node 1: {e}")
        return {
            "extracted_skills": "Could not analyze CV",
            "difficulty": "medium"
        }

print("✅ Node 1 ready!")

✅ Node 1 ready!


In [9]:
def retrieve_knowledge(state: InterviewState):
    print("🔍 Node 2: Retrieving knowledge...")
    try:
        query = f"Interview questions for: {state['extracted_skills']}"
        docs = retriever.invoke(query)
        context = "\n\n".join([doc.page_content for doc in docs])
        print(f"📄 Retrieved {len(docs)} relevant chunks!")
        return {"retrieved_context": context}
    except Exception as e:
        print(f"❌ Error in Node 2: {e}")
        return {"retrieved_context": state['jd_text']}

print("✅ Node 2 ready!")

✅ Node 2 ready!


In [10]:
def generate_question(state: InterviewState):
    print("🎯 Node 3: Generating question...")
    print(f"📊 Current difficulty: {state['difficulty']}")
    try:
        prompt = f"""
        You are a senior technical interviewer.
        Generate ONE interview question based on:

        Context: {state['retrieved_context']}
        Candidate skills: {state['extracted_skills']}
        Difficulty level: {state['difficulty']}
        Already asked questions: {state['asked_questions']}

        Rules:
        - easy = beginner level question
        - medium = intermediate level question
        - hard = advanced level question
        - Generate ONLY ONE question
        - Do NOT repeat previous questions
        - Return ONLY the question, nothing else
        """
        response = llm.invoke(prompt)
        return {"current_question": response.content}
    except Exception as e:
        print(f"❌ Error in Node 3: {e}")
        return {"current_question": "Tell me about your experience?"}

print("✅ Node 3 ready!")

✅ Node 3 ready!


In [11]:
def evaluate_answer(state: InterviewState):
    print("👔 Node 4: Evaluating answer...")

    # Get user answer - outside try/except so we always get input
    user_answer = input(f"\n🎤 Question: {state['current_question']}\n\nYour answer: ")

    try:
        # Evaluation prompt
        prompt = f"""
        Question: {state['current_question']}
        User answer: {user_answer}
        Job requirements: {state['jd_text']}

        Evaluate and reply in EXACTLY this format:
        SCORE: [number 1-10]
        FEEDBACK: [detailed feedback]
        """

        evaluation = llm.invoke(prompt)
        score = 0
        feedback = ""

        for line in evaluation.content.split("\n"):
            if "SCORE:" in line:
                try:
                    score = int(''.join(filter(str.isdigit, line.split("SCORE:")[1])))
                except:
                    score = 5
            if "FEEDBACK:" in line:
                feedback = line.split("FEEDBACK:")[1].strip()

        if score >= 8:
            difficulty = "hard"
        elif score >= 5:
            difficulty = "medium"
        else:
            difficulty = "easy"

        hint = ""
        if score < 5:
            hint_prompt = f"""
            As an expert interviewer give a helpful hint for:
            {state['current_question']}
            Do not reveal the answer.
            """
            hint = llm.invoke(hint_prompt).content

        if score >= 5:
            print(f"\n✅ Score: {score}/10")
            print(f"💬 Feedback: {feedback}")
        else:
            print(f"\n❌ Score: {score}/10")
            print(f"💬 Feedback: {feedback}")
            print(f"💡 Hint: {hint}")

        return {
            "user_answer": user_answer,
            "score": score,
            "feedback": feedback,
            "hint": hint,
            "difficulty": difficulty,
            "question_count": state['question_count'] + 1,
            "total_score": state['total_score'] + score,
            "asked_questions": state['asked_questions'] + [state['current_question']],
            "messages": [HumanMessage(content=user_answer)]
        }

    except Exception as e:
        print(f"❌ Error in Node 4: {e}")
        return {
            "user_answer": user_answer,
            "score": 5,
            "feedback": "Could not evaluate",
            "hint": "",
            "difficulty": "medium",
            "question_count": state['question_count'] + 1,
            "total_score": state['total_score'] + 5,
            "asked_questions": state['asked_questions'] + [state['current_question']],
            "messages": [HumanMessage(content=user_answer)]
        }

print("✅ Node 4 ready!")

✅ Node 4 ready!


In [12]:
def generate_report(state: InterviewState):
    print("📊 Node 5: Generating final report...")
    try:
        prompt = f"""
        Generate a professional interview performance report:
        Candidate Skills: {state['extracted_skills']}
        Total Score: {state['total_score']} out of {state['question_count'] * 10}
        Questions Asked: {state['asked_questions']}
        Last Feedback: {state['feedback']}
        Job Requirements: {state['jd_text']}
        """

        report = llm.invoke(prompt)

        print("\n" + "="*50)
        print("📄 FINAL INTERVIEW REPORT:")
        print("="*50)
        print(report.content)

    except Exception as e:
        print(f"❌ Error in Node 5: {e}")
        # In case of an error, report might not be defined yet, so initialize it
        report = type('obj', (object,), {'content': f"Error generating report: {e}"})()
    return {"final_report": report.content}

print("✅ Node 5 ready!")

✅ Node 5 ready!


Graph


In [13]:
# Condition function
def should_continue(state: InterviewState):
    print(f"🔄 Questions completed: {state['question_count']}/5")
    if state['question_count'] >= 5:
        print("➡️ Going to report!")
        return "generate_report"
    else:
        print("➡️ Next question!")
        return "retrieve_knowledge"

# Build graph
graph = StateGraph(InterviewState)

# Add 5 nodes
graph.add_node("analyze_resume", analyze_resume)
graph.add_node("retrieve_knowledge", retrieve_knowledge)
graph.add_node("generate_questions", generate_question)
graph.add_node("evaluate_answer", evaluate_answer)
graph.add_node("generate_report", generate_report)

# Entry point
graph.set_entry_point("analyze_resume")

# Normal edges
graph.add_edge("analyze_resume", "retrieve_knowledge")
graph.add_edge( "retrieve_knowledge", "generate_questions")
graph.add_edge("generate_questions", "evaluate_answer")

# Conditional edge
graph.add_conditional_edges(
    "evaluate_answer",
    should_continue,
    {
        "generate_report": "generate_report",
        "retrieve_knowledge": "retrieve_knowledge"
    }
)

# End
graph.add_edge("generate_report", END)

# Compile
app = graph.compile()
print("✅ Graph ready!")

✅ Graph ready!


In [14]:
# Initial state
initial_state = {
    "cv_text": cv_text,
    "jd_text": jd_text,
    "extracted_skills": "",
    "retrieved_context": "",
    "current_question": "",
    "user_answer": "",
    "score": 0,
    "feedback": "",
    "hint": "",
    "difficulty": "medium",
    "question_count": 0,
    "total_score": 0,
    "asked_questions": [],
    "final_report": "",
    "messages": []
}

print("🚀 Starting AI Interview Copilot V2!")
print("="*50)

result = app.invoke(initial_state)

🚀 Starting AI Interview Copilot V2!
📄 Node 1: Analyzing resume...
🔍 Node 2: Retrieving knowledge...
📄 Retrieved 2 relevant chunks!
🎯 Node 3: Generating question...
📊 Current difficulty: medium
👔 Node 4: Evaluating answer...

🎤 Question: Suppose you are tasked with analyzing customer purchase behavior for an e-commerce company, and you have a dataset containing customer demographics, purchase history, and product information. Write a Python code snippet using SQL queries to extract the top 5 products with the highest average order value, and then use Tableau or Power BI to create a visualization that displays the relationship between customer age and average order value, what would be your approach?

Your answer: abc

❌ Score: 1/10
💬 Feedback: The user's response, "abc", does not demonstrate any understanding of the task or the requirements. The task requires a Python code snippet using SQL queries to extract the top 5 products with the highest average order value, and then use Tableau 